# Experimental Model 2 — BiLSTM

**Task 2 — Yelp Polarity Sentiment Classification**

This notebook trains and evaluates **Experimental Model 2 (Bidirectional LSTM)**.
It reuses the same preprocessing, vocabulary, sequence length, and splits as the
baseline and TextCNN notebooks.

## Architecture & embedding justification

- **Change vs baseline / TextCNN:** replace bag-of-embeddings / local CNN
  filters with a **bidirectional LSTM** that models left-to-right and
  right-to-left context.
- **Why BiLSTM:** sentiment often depends on long-range cues ("... was great
  until ..." / delayed negation). Recurrence can carry that context across the
  review better than mean-pooling or fixed n-gram kernels.
- **Embeddings:** randomly initialized `nn.Embedding` learned from scratch
  (no pretrained embeddings / LMs).
- **Output:** single logit + `BCEWithLogitsLoss` (same setup as baseline).

## Hardware used for the reported run

- **GPU:** NVIDIA GeForce RTX 5090
- **Peak GPU memory:** ~224.5 MB
- **Training time:** ~583.6 s (5 epochs)
- **Throughput:** ~4,318 examples/sec

## 0. Setup

In [1]:
import os
from pathlib import Path
import re
import json
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    brier_score_loss,
)
from statsmodels.stats.contingency_tables import mcnemar
from tqdm.auto import tqdm

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

# Member folder = parent of src/ when cwd is src; otherwise cwd itself.
_cwd = Path.cwd().resolve()
BASE_DIR = str(_cwd.parent if _cwd.name == 'src' else _cwd)
os.makedirs(f"{BASE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{BASE_DIR}/logs", exist_ok=True)
os.makedirs(f"{BASE_DIR}/outputs", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("BASE_DIR:", BASE_DIR)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

BASE_DIR: .
Device: cuda
GPU: NVIDIA GeForce RTX 4090


## 1. Data load + identical preprocessing as baseline

In [2]:
dataset = load_dataset("fancyzhx/yelp_polarity")

train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

train_df = train_df.dropna(subset=["text", "label"]).copy()
test_df = test_df.dropna(subset=["text", "label"]).copy()
train_df = train_df[train_df["text"].astype(str).str.strip().ne("")].reset_index(drop=True)
test_df = test_df[test_df["text"].astype(str).str.strip().ne("")].reset_index(drop=True)

print(train_df.shape, test_df.shape)
print(train_df["label"].value_counts().sort_index())

(560000, 2) (38000, 2)
label
0    280000
1    280000
Name: count, dtype: int64


In [3]:
stop_words = set(stopwords.words("english"))
negation_words = {"no", "not", "nor", "never"}
stop_words = stop_words - negation_words
lemmatizer = WordNetLemmatizer()


def preprocess_text(text):
    text = text.lower()
    text = text.replace("\\n", " ").replace("\\r", " ")
    text = text.replace("\n", " ").replace("\r", " ")

    text = re.sub(r"\bwon't\b", "will not", text)
    text = re.sub(r"\bcan't\b", "can not", text)
    text = re.sub(r"n't\b", " not", text)

    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words
    ]
    return tokens

In [4]:
train_split, val_split = train_test_split(
    train_df,
    test_size=0.10,
    random_state=42,
    stratify=train_df["label"],
)
train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)
test_split = test_df.reset_index(drop=True)

for split_name, split_df in [
    ("train", train_split),
    ("val", val_split),
    ("test", test_split),
]:
    split_df["tokens"] = [
        preprocess_text(t) for t in tqdm(split_df["text"], desc=f"Preprocess {split_name}")
    ]

print(len(train_split), len(val_split), len(test_split))

Preprocess train:   0%|          | 0/504000 [00:00<?, ?it/s]

Preprocess val:   0%|          | 0/56000 [00:00<?, ?it/s]

Preprocess test:   0%|          | 0/38000 [00:00<?, ?it/s]

504000 56000 38000


In [5]:
VOCAB_SIZE = 30000
MAX_LEN = 160
PAD_TOKEN, UNK_TOKEN = "<PAD>", "<UNK>"
PAD_IDX, UNK_IDX = 0, 1

word_counts = Counter()
for tokens in tqdm(train_split["tokens"], desc="Building vocabulary"):
    word_counts.update(tokens)

word_to_idx = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
for word, _ in word_counts.most_common(VOCAB_SIZE - 2):
    word_to_idx[word] = len(word_to_idx)

print("Vocab size:", len(word_to_idx))

Building vocabulary:   0%|          | 0/504000 [00:00<?, ?it/s]

Vocab size: 30000


In [6]:
class YelpDataset(Dataset):
    def __init__(self, dataframe, word_to_idx, max_len):
        self.tokens = dataframe["tokens"].tolist()
        self.labels = dataframe["label"].tolist()
        self.word_to_idx = word_to_idx
        self.max_len = max_len
        self.pad_idx = word_to_idx["<PAD>"]
        self.unk_idx = word_to_idx["<UNK>"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        token_ids = [
            self.word_to_idx.get(token, self.unk_idx)
            for token in self.tokens[idx]
        ]
        if len(token_ids) == 0:
            token_ids = [self.unk_idx]

        token_ids = token_ids[: self.max_len]
        if len(token_ids) < self.max_len:
            token_ids += [self.pad_idx] * (self.max_len - len(token_ids))

        return (
            torch.tensor(token_ids, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float32),
        )


BATCH_SIZE = 128
train_dataset = YelpDataset(train_split, word_to_idx, MAX_LEN)
val_dataset = YelpDataset(val_split, word_to_idx, MAX_LEN)
test_dataset = YelpDataset(test_split, word_to_idx, MAX_LEN)

loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=0, pin_memory=torch.cuda.is_available())
train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print(len(train_dataset), len(val_dataset), len(test_dataset))

504000 56000 38000


## 2. BiLSTM model

In [7]:
class BiLSTMClassifier(nn.Module):
    """Bidirectional LSTM sentiment classifier (Experimental Model 2)."""

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_layers,
        padding_idx,
        dropout=0.5,
    ):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=padding_idx
        )
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, 1)

    def forward(self, input_ids):
        # Pack padded sequences so PAD tokens do not update the LSTM state
        mask = input_ids != self.embedding.padding_idx
        lengths = mask.sum(dim=1).clamp(min=1).cpu()

        embedded = self.embedding(input_ids)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths, batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)

        # h_n: [num_layers * 2, B, H] -> concat final forward/backward
        forward_last = h_n[-2]
        backward_last = h_n[-1]
        hidden = torch.cat([forward_last, backward_last], dim=1)
        hidden = self.dropout(hidden)
        return self.classifier(hidden).squeeze(1)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 1
DROPOUT = 0.5
LEARNING_RATE = 1e-3
EPOCHS = 5

model = BiLSTMClassifier(
    vocab_size=len(word_to_idx),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    padding_idx=PAD_IDX,
    dropout=DROPOUT,
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)
print("Trainable parameters:", f"{count_parameters(model):,}")

BiLSTMClassifier(
  (embedding): Embedding(30000, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (classifier): Linear(in_features=256, out_features=1, bias=True)
)
Trainable parameters: 4,104,449


## 3. Train / evaluate helpers

In [8]:
def evaluate_basic(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_labels, all_predictions = [], []

    with torch.no_grad():
        for input_ids, labels in loader:
            input_ids = input_ids.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(input_ids)
            loss = criterion(logits, labels)
            total_loss += loss.item() * input_ids.size(0)

            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).long()
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(preds.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_predictions)
    macro_f1 = f1_score(all_labels, all_predictions, average="macro")
    return avg_loss, acc, macro_f1


def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs,
    checkpoint_path,
    log_path,
):
    best_val_f1 = -1.0
    history = []

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    total_start = time.time()
    with open(log_path, "w", encoding="utf-8") as log_file:
        for epoch in range(1, epochs + 1):
            epoch_start = time.time()
            model.train()
            running_loss = 0.0
            examples_seen = 0

            for input_ids, labels in train_loader:
                input_ids = input_ids.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                optimizer.zero_grad()
                logits = model(input_ids)
                loss = criterion(logits, labels)
                loss.backward()
                optimizer.step()

                bs = input_ids.size(0)
                running_loss += loss.item() * bs
                examples_seen += bs

            train_loss = running_loss / examples_seen
            val_loss, val_acc, val_f1 = evaluate_basic(
                model, val_loader, criterion, device
            )
            epoch_time = time.time() - epoch_start
            examples_per_sec = examples_seen / epoch_time

            row = {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_accuracy": val_acc,
                "val_macro_f1": val_f1,
                "epoch_time_sec": epoch_time,
                "examples_per_sec": examples_per_sec,
            }
            history.append(row)

            line = (
                f"Epoch {epoch}/{epochs} | Train Loss: {train_loss:.4f} | "
                f"Val Loss: {val_loss:.4f} | Val Accuracy: {val_acc:.4f} | "
                f"Val Macro-F1: {val_f1:.4f} | Time: {epoch_time:.2f}s | "
                f"Examples/sec: {examples_per_sec:.2f}"
            )
            print(line)
            log_file.write(line + "\n")
            log_file.flush()

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                torch.save(
                    {
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "epoch": epoch,
                        "val_macro_f1": val_f1,
                        "max_len": MAX_LEN,
                        "vocab_size": len(word_to_idx),
                        "embedding_dim": EMBEDDING_DIM,
                        "hidden_dim": HIDDEN_DIM,
                        "num_layers": NUM_LAYERS,
                        "dropout": DROPOUT,
                    },
                    checkpoint_path,
                )
                print(f"Best checkpoint saved (Macro-F1={val_f1:.4f})")

    total_time = time.time() - total_start
    peak_mem = (
        torch.cuda.max_memory_allocated() / (1024 ** 2)
        if torch.cuda.is_available()
        else 0.0
    )
    return history, total_time, peak_mem

## 4. Train BiLSTM

Set `RETRAIN = False` to reload the existing best checkpoint instead of training again.

In [9]:
RETRAIN = False  # True = train from scratch; False = load saved checkpoint if present

checkpoint_path = f"{BASE_DIR}/checkpoints/bilstm_experimental2_best.pt"
log_path = f"{BASE_DIR}/logs/bilstm_experimental2_raw_training.log"

if RETRAIN or not os.path.exists(checkpoint_path):
    history, training_time, peak_memory = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        epochs=EPOCHS,
        checkpoint_path=checkpoint_path,
        log_path=log_path,
    )
    pd.DataFrame(history).to_csv(
        f"{BASE_DIR}/outputs/bilstm_experimental2_training_history.csv",
        index=False,
    )
else:
    print("Loading existing checkpoint:", checkpoint_path)
    summary_path = f"{BASE_DIR}/outputs/bilstm_experimental2_training_summary.json"
    if os.path.exists(summary_path):
        with open(summary_path, encoding="utf-8") as f:
            prev = json.load(f)
        training_time = prev.get("training_time_sec", float("nan"))
        peak_memory = prev.get("peak_gpu_memory_mb", float("nan"))
        history = []
        print("Restored prior training_time_sec / peak_gpu_memory_mb from summary.")
    else:
        training_time, peak_memory, history = float("nan"), float("nan"), []

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Loaded epoch:", checkpoint.get("epoch"))
print("Best val Macro-F1:", checkpoint.get("val_macro_f1"))

Loading existing checkpoint: ./checkpoints/bilstm_experimental2_best.pt
Restored prior training_time_sec / peak_gpu_memory_mb from summary.
Loaded epoch: 5
Best val Macro-F1: 0.9464426720269712


## 5. Full test metrics

In [10]:
def expected_calibration_error(y_true, y_prob, n_bins=15):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    bin_boundaries = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob > bin_boundaries[i]) & (y_prob <= bin_boundaries[i + 1])
        if mask.sum() == 0:
            continue
        conf = y_prob[mask].mean()
        acc = y_true[mask].mean()
        ece += (mask.mean()) * abs(acc - conf)
    return float(ece)


def bootstrap_confidence_intervals(y_true, y_pred, n_bootstrap=1000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    accs, f1s, mccs = [], [], []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        yt, yp = y_true[idx], y_pred[idx]
        accs.append(accuracy_score(yt, yp))
        f1s.append(f1_score(yt, yp, average="macro"))
        mccs.append(matthews_corrcoef(yt, yp))
    def ci(vals):
        return [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))]
    return {"accuracy": ci(accs), "macro_f1": ci(f1s), "mcc": ci(mccs)}


all_labels, all_probs, all_preds = [], [], []
model.eval()
with torch.no_grad():
    for input_ids, labels in test_loader:
        input_ids = input_ids.to(device, non_blocking=True)
        logits = model(input_ids)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).long()
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

y_true = np.array(all_labels)
y_prob = np.array(all_probs)
y_pred = np.array(all_preds)

accuracy = accuracy_score(y_true, y_pred)
precision_macro = precision_score(y_true, y_pred, average="macro")
precision_micro = precision_score(y_true, y_pred, average="micro")
precision_weighted = precision_score(y_true, y_pred, average="weighted")
recall_macro = recall_score(y_true, y_pred, average="macro")
recall_micro = recall_score(y_true, y_pred, average="micro")
recall_weighted = recall_score(y_true, y_pred, average="weighted")
f1_macro = f1_score(y_true, y_pred, average="macro")
f1_micro = f1_score(y_true, y_pred, average="micro")
f1_weighted = f1_score(y_true, y_pred, average="weighted")
cm = confusion_matrix(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_prob)
pr_auc = average_precision_score(y_true, y_prob)
mcc = matthews_corrcoef(y_true, y_pred)
brier = brier_score_loss(y_true, y_prob)
ece = expected_calibration_error(y_true, y_prob)

print("=" * 50)
print("BILSTM EXPERIMENTAL2 TEST RESULTS")
print("=" * 50)
print(f"Accuracy:             {accuracy:.4f}")
print(f"Precision Macro:      {precision_macro:.4f}")
print(f"Precision Micro:      {precision_micro:.4f}")
print(f"Precision Weighted:   {precision_weighted:.4f}")
print(f"Recall Macro:         {recall_macro:.4f}")
print(f"Recall Micro:         {recall_micro:.4f}")
print(f"Recall Weighted:      {recall_weighted:.4f}")
print(f"F1 Macro:             {f1_macro:.4f}")
print(f"F1 Micro:             {f1_micro:.4f}")
print(f"F1 Weighted:          {f1_weighted:.4f}")
print(f"ROC-AUC:              {roc_auc:.4f}")
print(f"PR-AUC:               {pr_auc:.4f}")
print(f"MCC:                   {mcc:.4f}")
print(f"Brier Score:           {brier:.4f}")
print(f"ECE:                   {ece:.4f}")
print("Confusion Matrix:\n", cm)

BILSTM EXPERIMENTAL2 TEST RESULTS
Accuracy:             0.9486
Precision Macro:      0.9487
Precision Micro:      0.9486
Precision Weighted:   0.9487
Recall Macro:         0.9486
Recall Micro:         0.9486
Recall Weighted:      0.9486
F1 Macro:             0.9486
F1 Micro:             0.9486
F1 Weighted:          0.9486
ROC-AUC:              0.9884
PR-AUC:               0.9885
MCC:                   0.8974
Brier Score:           0.0401
ECE:                   0.0211
Confusion Matrix:
 [[18157   843]
 [ 1109 17891]]


In [11]:
bootstrap_ci = bootstrap_confidence_intervals(y_true, y_pred)
print("95% Bootstrap CIs")
print("Accuracy:", bootstrap_ci["accuracy"])
print("Macro-F1:", bootstrap_ci["macro_f1"])
print("MCC:", bootstrap_ci["mcc"])

95% Bootstrap CIs
Accuracy: [0.9463947368421053, 0.9507901315789473]
Macro-F1: [0.946391491173435, 0.950785710659793]
MCC: [0.892902502263747, 0.9016641504443382]


In [12]:
slice_negation_words = {"no", "not", "nor", "never", "none", "nobody", "nothing"}
token_lengths = test_split["tokens"].apply(len).to_numpy()
has_negation = test_split["tokens"].apply(
    lambda toks: any(t in slice_negation_words for t in toks)
).to_numpy()

slice_masks = {
    "Short (<=50 tokens)": token_lengths <= 50,
    "Medium (51-144 tokens)": (token_lengths > 50) & (token_lengths <= 144),
    "Long (>144 tokens)": token_lengths > 144,
    "Contains Negation": has_negation,
    "No Explicit Negation": ~has_negation,
}


def evaluate_slices(y_true, y_pred, slice_masks):
    rows = []
    for name, mask in slice_masks.items():
        yt, yp = y_true[mask], y_pred[mask]
        rows.append(
            {
                "slice": name,
                "n_examples": int(mask.sum()),
                "macro_f1": f1_score(yt, yp, average="macro") if len(yt) else float("nan"),
                "error_rate": float((yt != yp).mean()) if len(yt) else float("nan"),
            }
        )
    return pd.DataFrame(rows)


slice_results = evaluate_slices(y_true, y_pred, slice_masks)
print("BILSTM ROBUSTNESS BY DATA SLICE")
print(slice_results.to_string(index=False))

BILSTM ROBUSTNESS BY DATA SLICE
                 slice  n_examples  macro_f1  error_rate
   Short (<=50 tokens)       19274  0.952396    0.046903
Medium (51-144 tokens)       15046  0.949324    0.050179
    Long (>144 tokens)        3680  0.912301    0.079620
     Contains Negation       28293  0.945123    0.052875
  No Explicit Negation        9707  0.934750    0.046976


## 6. Save artifacts (including previously missing metrics.json)

In [13]:
param_count = count_parameters(model)
avg_eps = (
    float(np.mean([h["examples_per_sec"] for h in history]))
    if history
    else float("nan")
)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

metrics = {
    "model": "bilstm_experimental2",
    "accuracy": float(accuracy),
    "precision_macro": float(precision_macro),
    "precision_micro": float(precision_micro),
    "precision_weighted": float(precision_weighted),
    "recall_macro": float(recall_macro),
    "recall_micro": float(recall_micro),
    "recall_weighted": float(recall_weighted),
    "f1_macro": float(f1_macro),
    "f1_micro": float(f1_micro),
    "f1_weighted": float(f1_weighted),
    "roc_auc": float(roc_auc),
    "pr_auc": float(pr_auc),
    "mcc": float(mcc),
    "brier_score": float(brier),
    "ece": float(ece),
    "confusion_matrix": cm.tolist(),
    "parameter_count": int(param_count),
    "training_time_sec": float(training_time) if training_time == training_time else None,
    "peak_gpu_memory_mb": float(peak_memory) if peak_memory == peak_memory else None,
    "gpu": gpu_name,
}

summary = {
    "model": "bilstm_experimental2",
    "parameter_count": int(param_count),
    "training_time_sec": float(training_time) if training_time == training_time else None,
    "average_examples_per_sec": avg_eps if avg_eps == avg_eps else None,
    "peak_gpu_memory_mb": float(peak_memory) if peak_memory == peak_memory else None,
    "gpu": gpu_name,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
}

preds_df = pd.DataFrame(
    {
        "true_label": y_true.astype(int),
        "predicted_label": y_pred.astype(int),
        "positive_probability": y_prob.astype(float),
    }
)
preds_df.to_csv(f"{BASE_DIR}/outputs/bilstm_experimental2_predictions.csv", index=False)
pd.DataFrame(cm).to_csv(
    f"{BASE_DIR}/outputs/bilstm_experimental2_confusion_matrix.csv", index=False
)
slice_results.to_csv(
    f"{BASE_DIR}/outputs/bilstm_experimental2_slice_results.csv", index=False
)

with open(f"{BASE_DIR}/outputs/bilstm_experimental2_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
with open(f"{BASE_DIR}/outputs/bilstm_experimental2_training_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
with open(f"{BASE_DIR}/outputs/bilstm_experimental2_bootstrap_ci.json", "w", encoding="utf-8") as f:
    json.dump(bootstrap_ci, f, indent=2)

print("Saved BiLSTM outputs to", f"{BASE_DIR}/outputs")

Saved BiLSTM outputs to ./outputs


## 7. Paired McNemar tests (baseline vs each experimental)

Uses saved prediction CSVs so this section works even if you only re-run evaluation.

In [16]:
def run_mcnemar(y_true, y_pred_a, y_pred_b, name_a, name_b):
    y_true = np.asarray(y_true).astype(int)
    a = np.asarray(y_pred_a).astype(int)
    b = np.asarray(y_pred_b).astype(int)

    # Contingency for paired correctness
    a_correct = a == y_true
    b_correct = b == y_true
    n01 = int(((~a_correct) & b_correct).sum())  # A wrong, B right
    n10 = int((a_correct & (~b_correct)).sum())  # A right, B wrong
    table = [[0, n01], [n10, 0]]

    result = mcnemar(table, exact=False, correction=True)
    print(f"McNemar: {name_a} vs {name_b}")
    print(f"  A wrong / B right (n01): {n01}")
    print(f"  A right / B wrong (n10): {n10}")
    print(f"  statistic={result.statistic:.4f}, p-value={result.pvalue:.6g}")
    return {"n01": n01, "n10": n10, "statistic": float(result.statistic), "pvalue": float(result.pvalue)}


baseline_pred_path = f"{BASE_DIR}/outputs/baseline_predictions.csv"
textcnn_pred_path = f"{BASE_DIR}/outputs/textcnn_experimental1_predictions.csv"
bilstm_pred_path = f"{BASE_DIR}/outputs/bilstm_experimental2_predictions.csv"

# Prefer freshly computed BiLSTM preds; fall back to CSV
bilstm_preds = y_pred
bilstm_true = y_true

mcnemar_results = {}

if os.path.exists(baseline_pred_path) and os.path.exists(textcnn_pred_path):
    base = pd.read_csv(baseline_pred_path)
    cnn = pd.read_csv(textcnn_pred_path)
    assert len(base) == len(cnn)
    mcnemar_results["baseline_vs_textcnn"] = run_mcnemar(
        base["true_label"],
        base["predicted_label"],
        cnn["predicted_label"],
        "baseline",
        "textcnn",
    )
else:
    print("Missing baseline and/or TextCNN prediction CSV — skip baseline vs TextCNN.")
    print("Expected:", baseline_pred_path)

if os.path.exists(baseline_pred_path):
    base = pd.read_csv(baseline_pred_path)
    mcnemar_results["baseline_vs_bilstm"] = run_mcnemar(
        base["true_label"],
        base["predicted_label"],
        bilstm_preds,
        "baseline",
        "bilstm",
    )
else:
    print("Missing baseline prediction CSV — cannot run McNemar vs BiLSTM yet.")
    print("Export baseline predictions from baseline.ipynb to:", baseline_pred_path)

with open(f"{BASE_DIR}/outputs/mcnemar_results.json", "w", encoding="utf-8") as f:
    json.dump(mcnemar_results, f, indent=2)

McNemar: baseline vs textcnn
  A wrong / B right (n01): 1057
  A right / B wrong (n10): 854
  statistic=21.3522, p-value=3.82188e-06
McNemar: baseline vs bilstm
  A wrong / B right (n01): 1212
  A right / B wrong (n10): 663
  statistic=160.1621, p-value=1.04286e-36


## 8. Manual error review template (BiLSTM — best model)

Assignment asks for **20** reviewed errors:
5 confident FP, 5 confident FN, 5 near-threshold, 5 slice-specific.

This cell samples candidates so you can label each with an error type + one
testable fix in the markdown table below.

In [15]:
review_df = test_split.copy()
review_df["true_label"] = y_true
review_df["predicted_label"] = y_pred
review_df["positive_probability"] = y_prob
review_df["token_len"] = review_df["tokens"].apply(len)
review_df["error"] = review_df["true_label"] != review_df["predicted_label"]

errors = review_df[review_df["error"]].copy()

confident_fp = errors[
    (errors["true_label"] == 0) & (errors["positive_probability"] >= 0.90)
].nlargest(5, "positive_probability")

confident_fn = errors[
    (errors["true_label"] == 1) & (errors["positive_probability"] <= 0.10)
].nsmallest(5, "positive_probability")

near_threshold = errors[
    (errors["positive_probability"] - 0.5).abs() <= 0.05
].copy()
near_threshold["dist"] = (near_threshold["positive_probability"] - 0.5).abs()
near_threshold = near_threshold.nsmallest(5, "dist")

slice_specific = errors[errors["token_len"] > 144].copy()
if len(slice_specific) < 5:
    slice_specific = errors.copy()
slice_specific = slice_specific.head(5)


def show_cases(df, title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    for i, row in df.iterrows():
        print(f"\nidx={i} true={int(row['true_label'])} pred={int(row['predicted_label'])} "
              f"p={row['positive_probability']:.4f} len={row['token_len']}")
        print(str(row["text"])[:500].replace("\n", " "))


show_cases(confident_fp, "Confident False Positives (p>=0.90, true=0)")
show_cases(confident_fn, "Confident False Negatives (p<=0.10, true=1)")
show_cases(near_threshold, "Near-threshold errors (|p-0.5|<=0.05)")
show_cases(slice_specific, "Slice-specific failures (long reviews)")

error_review_export = pd.concat(
    [
        confident_fp.assign(bucket="confident_fp"),
        confident_fn.assign(bucket="confident_fn"),
        near_threshold.assign(bucket="near_threshold"),
        slice_specific.assign(bucket="slice_long"),
    ]
)
error_review_export[[
    "bucket", "true_label", "predicted_label", "positive_probability", "token_len", "text"
]].to_csv(f"{BASE_DIR}/outputs/bilstm_error_review_candidates.csv", index=True)
print("\nSaved candidates to outputs/bilstm_error_review_candidates.csv")


Confident False Positives (p>=0.90, true=0)

idx=29330 true=0 pred=1 p=1.0000 len=18
Wow love the place and everything is very clean and new!\n\nGreat place to come and relax worth a try!\n\nCheers,\n\nEric Van Nguyen\nVisited April 2012

idx=27375 true=0 pred=1 p=0.9999 len=47
We had dinner here for four. Had an artichoke appetizer with pita chips, club sandwich and shared a huge sundae for dessert. The service was awesome; space impressive. But, I have to say that when a gem from NYC gets transported to Vegas, and gets \""blown up\"" into Vegas standards with all the glam, size, \""in your face\"", and over-the-top stuff that makes Vegas what it is, that NYC gem looses something and just isn't the same. \n\nWe were looking to be awed by this chocolate place and weren

idx=10515 true=0 pred=1 p=0.9999 len=39
Honestly my fav place for buffet at Vegas is the Wynn....similar in price but the food served here somehow for me missed it's mark.....and I am more into main meal as opposed to 

### Error review table (fill after reading the 20 candidates)

| # | Bucket | Error type | Testable fix |
|---|---|---|---|
| 1 | Confident FP | e.g. sarcasm / mixed sentiment | Keep contrastive cues; add contrastive-phrase features or longer context |
| 2 | Confident FP | | |
| 3 | Confident FP | | |
| 4 | Confident FP | | |
| 5 | Confident FP | | |
| 6 | Confident FN | e.g. negation scope | Preserve more negation context / dependency-aware features |
| 7 | Confident FN | | |
| 8 | Confident FN | | |
| 9 | Confident FN | | |
| 10 | Confident FN | | |
| 11 | Near-threshold | calibration / ambiguous | Temperature scaling or threshold tuning on val set |
| 12 | Near-threshold | | |
| 13 | Near-threshold | | |
| 14 | Near-threshold | | |
| 15 | Near-threshold | | |
| 16 | Slice (long) | truncation at MAX_LEN=160 | Increase MAX_LEN / hierarchical pooling over chunks |
| 17 | Slice (long) | | |
| 18 | Slice (long) | | |
| 19 | Slice (long) | | |
| 20 | Slice (long) | | |

## Reported results from the completed run (already in `outputs/`)

| Metric | Value |
|---|---|
| Accuracy | **0.9486** |
| Bootstrap Acc 95% CI | [0.9464, 0.9508] |
| MCC | ~0.897 |
| Params | 4,104,449 |
| Train time | 583.6 s |
| Peak GPU mem | 224.5 MB |
| GPU | NVIDIA GeForce RTX 5090 |

BiLSTM is currently the best of the three models on held-out test accuracy.